In [3]:
!pip install torch torchvision pandas numpy pillow scikit-learn

In [4]:
!pip install pandas numpy PIL

ERROR: Could not find a version that satisfies the requirement PIL (from versions: none)
ERROR: No matching distribution found for PIL


In [7]:
import os
import torch
import numpy as np
import pandas as pd
from PIL import Image
from torchvision import transforms
from torch.utils.data import DataLoader, Dataset, Subset

# Swin-Tiny requires 224x224 input and ImageNet Normalization
def get_transforms():
    """
    Defines the data augmentations and standardizations for the dataset.
    """
    # Standard image dimensions for models like ResNet/DenseNet
    IMAGE_SIZE = (224, 224)

    # ImageNet statistics for normalization
    MEAN = [0.485, 0.456, 0.406]
    STD = [0.229, 0.224, 0.225]

    train_transform = transforms.Compose([
        transforms.Resize(IMAGE_SIZE),
        transforms.RandomHorizontalFlip(), # Basic augmentation
        transforms.RandomVerticalFlip(),   # Basic augmentation
        transforms.ToTensor(),
        transforms.Normalize(mean=MEAN, std=STD)
    ])

    test_transform = transforms.Compose([
        transforms.Resize(IMAGE_SIZE),
        transforms.ToTensor(),
        transforms.Normalize(mean=MEAN, std=STD)
    ])

    return train_transform, test_transform

In [8]:
class ISICMilkDataset(Dataset):
    def __init__(self, csv_file, img_dir, transform=None):
        self.labels_frame = pd.read_csv(csv_file)
        self.img_dir = img_dir
        self.transform = transform
        
        # The 11 class columns from your ground truth
        self.class_names = ['AKIEC', 'BCC', 'BEN_OTH', 'BKL', 'DF', 'INF', 'MAL_OTH', 'MEL', 'NV', 'SCCKA', 'VASC']

    def __len__(self):
        return len(self.labels_frame)

    def __getitem__(self, idx):
        row = self.labels_frame.iloc[idx]
        img_id = str(row['lesion_id'])
        
        # Base path pointing to the specific lesion folder
        base_path = os.path.join(self.img_dir, img_id)
        img_path = None
        
        # Check inside the folder for the actual image file
        if os.path.isdir(base_path):
            files = os.listdir(base_path)
            img_files = [f for f in files if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
            if img_files:
                img_path = os.path.join(base_path, img_files[0])
            else:
                raise FileNotFoundError(f"No image found inside: {base_path}")
        elif os.path.exists(base_path + ".jpg"):
            img_path = base_path + ".jpg"
        else:
            raise FileNotFoundError(f"Path does not exist: {base_path}")

        # Load image
        image = Image.open(img_path).convert('RGB')
        if self.transform:
            image = self.transform(image)
            
        # Convert the 11 one-hot columns into a single integer label (0 to 10)
        one_hot_vals = row[self.class_names].values.astype(float)
        label = int(np.argmax(one_hot_vals))

        return image, label

In [9]:
def dirichlet_split_noniid(train_labels, alpha, n_clients):
    n_classes = len(np.unique(train_labels))
    client_idcs = [[] for _ in range(n_clients)]
    
    for k in range(n_classes):
        idx_k = np.where(train_labels == k)[0]
        np.random.shuffle(idx_k)
        
        proportions = np.random.dirichlet(np.repeat(alpha, n_clients))
        proportions = np.array([p * (len(idx_j) < (len(train_labels) / n_clients)) for p, idx_j in zip(proportions, client_idcs)])
        proportions = proportions / proportions.sum()
        proportions = (np.cumsum(proportions) * len(idx_k)).astype(int)[:-1]
        
        client_idcs = [idx_j + idx.tolist() for idx_j, idx in zip(client_idcs, np.split(idx_k, proportions))]
        
    return client_idcs

def prepare_federated_data(train_csv, train_dir, test_csv, test_dir, n_clients=5, batch_size=32, alpha=0.5):
    train_transform, test_transform = get_transforms()
    
    print("Loading datasets... (This maps all file paths into memory)")
    train_dataset = ISICMilkDataset(train_csv, train_dir, transform=train_transform)
    test_dataset = ISICMilkDataset(test_csv, test_dir, transform=test_transform)
    
    # Extract labels to compute the heterogeneous split
    print("Computing non-IID Dirichlet split...")
    train_labels = np.array([np.argmax(train_dataset.labels_frame.iloc[i][train_dataset.class_names].values.astype(float)) for i in range(len(train_dataset))])
    
    client_indices = dirichlet_split_noniid(train_labels, alpha=alpha, n_clients=n_clients)
    
    client_loaders = []
    for indices in client_indices:
        subset = Subset(train_dataset, indices)
        loader = DataLoader(subset, batch_size=batch_size, shuffle=True, drop_last=True)
        client_loaders.append(loader)
        
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)
    
    print(f"Created {n_clients} FedDC client loaders.")
    print(f"Test loader ready with {len(test_dataset)} images.")
    
    # Save indices to disk
    torch.save(client_indices, 'federated_client_indices.pt')
    
    return client_loaders, test_loader, test_dataset

# --- EXECUTION ---
# Ensure these paths match where your script is running
TRAIN_CSV = 'Dataset/MILK10k_Training_GroundTruth.csv' # UPDATE if file name differs
TRAIN_DIR = 'Dataset/MILK10k_Training_Input/MILK10k_Training_Input/'

TEST_CSV = 'Dataset/MILK10k_Training_Metadata.csv'   # UPDATE if file name differs
TEST_DIR = 'Dataset/MILK10k_Test_Input/MILK10k_Test_Input/'

# Generate DataLoaders (Keep them in memory for File 2)
client_loaders, test_loader, test_dataset = prepare_federated_data(TRAIN_CSV, TRAIN_DIR, TEST_CSV, TEST_DIR, n_clients=5)

Loading datasets... (This maps all file paths into memory)
Computing non-IID Dirichlet split...
Created 5 FedDC client loaders.
Test loader ready with 10480 images.
